# Skin Lesion Bias Reduction — Colab training

Runs the EfficientNetV2-B0 baseline classifier on a Colab GPU using the project code in `src/` and a Fitzpatrick17k dataset you've already uploaded.

**Order of operations**
1. Confirm GPU + mount Drive (if used)
2. Point the notebook at your code + data
3. Install dependencies
4. Train the baseline (with class weights, unfreeze schedule, save-best-by-val-loss)
5. Evaluate the best checkpoint and render a markdown bias report
6. *Optional* — kick off the cGAN training (placeholder, run later)

## 1. Verify GPU and Colab environment

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Colab: True
torch: 2.10.0+cu128 cuda available: True
GPU: NVIDIA A100-SXM4-40GB
name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 40437 MiB


In [2]:
!nvidia-smi

Wed Apr 29 16:14:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             48W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Connect your code and data

Two common layouts work:

- **Drive layout** — repo and dataset both sit under `MyDrive`. Mount Drive and point `PROJECT_ROOT` at the repo there. Outputs persist between sessions.
- **Local Colab layout** — clone the repo into `/content/` and put the dataset under `/content/dataset/`. Faster I/O, but everything is wiped when the runtime ends.

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` in the cell below to match where you uploaded things.

In [3]:
# Mount Drive only if you're using the Drive layout. Skip this cell otherwise.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")  # repo root containing src/, dataset/, run_*.sh
# IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"

DATASET_CSV = PROJECT_ROOT / "dataset/fitzpatrick17k_cleaned.csv"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:   ", IMAGE_DIR,   "exists:", IMAGE_DIR.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"

%cd $PROJECT_ROOT

PROJECT_ROOT: /content/drive/MyDrive/SkinLesionBiasReduction
IMAGE_DIR:    /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images exists: True
/content/drive/MyDrive/SkinLesionBiasReduction


In [8]:
import subprocess, time  
from pathlib import Path

DRIVE_IMAGE_DIR = IMAGE_DIR
LOCAL_IMAGE_DIR = Path("/content/local_images")                                                                                                                                                
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)                                                                                                                                            

def _count_files(d):
    r = subprocess.run(f"ls -1 '{d}' 2>/dev/null | wc -l",
                     shell=True, capture_output=True, text=True)
    return int(r.stdout.strip() or 0)

n_source = _count_files(DRIVE_IMAGE_DIR)
n_local  = _count_files(LOCAL_IMAGE_DIR)
print(f"Drive: {n_source} files | Local: {n_local} files "
      f"(need ~{max(0, n_source - n_local)} more)")

if n_local >= n_source > 0:
    print("Local cache is complete — skipping copy.")
else:
    # Parallel copy: ~64 concurrent cp workers amortize Drive's per-file FUSE overhead.
    # `cp -n` = skip files that already exist at the destination, so this resumes
    # from any partial copy (no rm -rf needed).
    print("Parallel-copying from Drive (64 workers)...")
    t0 = time.time()
    cmd = (
        f"cd '{DRIVE_IMAGE_DIR}' && "
        f"find . -maxdepth 1 -type f -print0 | "
        f"xargs -0 -n 64 -P 64 cp -n -t '{LOCAL_IMAGE_DIR}/'"
    )
    subprocess.run(cmd, shell=True, check=True)
    n_local = _count_files(LOCAL_IMAGE_DIR)
    print(f"Done — {n_local} files in {time.time() - t0:.1f}s "
          f"({n_local / max(1, time.time() - t0):.0f} files/s)")
    assert n_local >= n_source, f"Copy incomplete: expected {n_source}, got {n_local}"

IMAGE_DIR = LOCAL_IMAGE_DIR
print(f"IMAGE_DIR → {IMAGE_DIR}")

Drive: 16518 files | Local: 16518 files (need ~0 more)
Local cache is complete — skipping copy.
IMAGE_DIR → /content/local_images


In [ ]:
# Alternative: if you don't have the repo on Drive, clone it into /content/ and copy your dataset in.
# Uncomment, replace the URL with your fork, and re-run cell `configure-paths` with PROJECT_ROOT=/content/SkinLesionBiasReduction.
# !git clone git@github.com:hoangnam310/SkinLesionBiasReduction.git

In [6]:
!git fetch origin main && git reset --hard origin/main

From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
Updating files: 100% (41/41), done.
HEAD is now at 1feb687 Add segmentation


## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, and `scipy`. The trainer additionally needs **timm** and **scikit-learn** (sklearn is usually preinstalled, timm usually is not). Tensorboard is optional and is also usually preinstalled.

In [7]:
!pip install --quiet timm 'scikit-learn>=1.3'

## 4. Train the baseline classifier

Full training at **224×224** for 40 epochs. The backbone is frozen for the first 3 epochs (head-only warmup), then unfrozen for fine-tuning at a lower LR.

Key flags being used:
- `--class_weights` — inverse-frequency weighted CrossEntropyLoss; the single biggest fix from the prior run's bias analysis.
- `--freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5` — three warmup epochs on the head, then full fine-tune.
- The trainer also auto-saves the best checkpoint by val_loss across all epochs (no extra flag needed).

> **Tip:** For a quick smoke test, set `EPOCHS = 1` and `IMAGE_SIZE = 64` to verify the pipeline works before committing to the full run.

In [6]:
IMAGE_SIZE   = 224
EPOCHS       = 40
BATCH_SIZE   = 32        # 32 fits comfortably on A100 at 224x224; use 16 on a T4.
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_efficientnet"
print("Outputs will land under:", OUTPUT_DIR)

Outputs will land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


In [33]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --image_size {IMAGE_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --weight_decay {WEIGHT_DECAY} \
    --num_workers {NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train batches: 399, Val batches: 100
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.4567, 2.4956, 0.4562]
Epoch [1/40] train_loss=1.0420, train_acc=0.4895, val_loss=1.0077, val_acc=0.5265  ← best
Epoch [2/40] train_loss=0.9842, train_acc=0.5554, val_loss=0.9713, val_acc=0.5591  ← best
Epoch [3/40] train_loss=0.9558, train_acc=0.5705, val_loss=0.9504, val_acc=0.5713  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9226, train_acc=0.5866, val_loss=0.9100, val_acc=0.5954  ← best
Epoch [5/40] train_loss=0.8769, train_acc=0.6133, val_loss=0.8795, val_acc=0.5926  ← best
Epoch [6/40] train_loss=0.8397, train_acc=0.6385, val_loss=0.8449, val_acc=0.6224  ← best
Epoch [7/40] train_loss=0.7976, train_acc=0.6636, val_loss=0.8191, val_acc=0.6509  ← best
Epoch [8/40] train_loss=0.7606, train_acc=0.6804, val_loss=0.79

## 5. Evaluate the best checkpoint

`train_baseline_efficientnet.py` restores best-by-val-loss weights before the final eval, so the auto-generated `metrics.json` already uses that snapshot. We additionally re-run `evaluate.py` to write `logs/evaluation_metrics.json` (with bias breakdowns) and then render a markdown report.

In [34]:
runs = sorted(OUTPUT_DIR.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No checkpoint.pt under {OUTPUT_DIR}"
LATEST_CKPT = runs[0]
print("Latest checkpoint:", LATEST_CKPT)

Latest checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260429_185247/checkpoint.pt


In [35]:
!python src/evaluate.py \
    --checkpoint "{LATEST_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260429_185247/checkpoint.pt
Evaluating on 3191 samples (100 batches)
Evaluation Summary
Samples: 3191
Top-1 Accuracy: 0.7202
Macro AUROC:    0.8521
Macro AUPRC:    0.7080

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=602): acc=0.6495 macroAUROC=0.8228 macroAUPRC=0.6678
             benign: n=  101 AUROC=0.7292 AUPRC=0.4002
          malignant: n=   85 AUROC=0.9076 AUPRC=0.7080
     non-neoplastic: n=  416 AUROC=0.8315 AUPRC=0.8954
fitzpatrick_2 (n=920): acc=0.7022 macroAUROC=0.8576 macroAUPRC=0.7173
             benign: n=  112 AUROC=0.8115 AUPRC=0.4770
          malignant: n=  164 AUROC=0.8976 AUPRC=0.7519
     non-neoplastic: n=  644 AUROC=0.8638 AUPRC=0.9231
fitzpatrick_3 (n=665): acc=0.7188 macroAUROC=0.8450 macroAUPRC=0.7166
             benign: n=   91 AUROC=0.7898 AUPRC=0.5106
          malignant:

In [15]:
!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

Wrote /content/drive/MyDrive/SkinLesionBiasReduction/logs/20260427_055032_report.md


In [16]:
from IPython.display import Markdown, display

run_name = LATEST_CKPT.parent.name
report_path = PROJECT_ROOT / "logs" / f"{run_name}_report.md"
print("Report:", report_path)
display(Markdown(report_path.read_text()))

Report: /content/drive/MyDrive/SkinLesionBiasReduction/logs/20260427_055032_report.md


# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_055032/checkpoint.pt` |
| Split | val |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_cleaned.csv` |
| Image dir | `/content/local_images` |
| Generated at | 2026-04-28T00:58:32.927764 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 3191 |
| Top-1 Accuracy | 0.7211 |
| Macro AUROC | 0.8521 |
| Macro AUPRC | 0.7080 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 602 | 0.6495 | 0.8228 | 0.6677 |
| 2 | 920 | 0.7054 | 0.8577 | 0.7174 |
| 3 | 665 | 0.7188 | 0.8449 | 0.7165 |
| 4 | 554 | 0.7834 | 0.8741 | 0.7506 |
| 5 | 318 | 0.7642 | 0.8625 | 0.7069 |
| 6 | 132 | 0.8030 | 0.8468 | 0.6530 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 101 / 0.7292 | 85 / 0.9077 | 416 / 0.8315 |
| 2 | 112 / 0.8115 | 164 / 0.8976 | 644 / 0.8638 |
| 3 | 91 / 0.7898 | 100 / 0.9000 | 474 / 0.8450 |
| 4 | 75 / 0.8525 | 59 / 0.9099 | 420 / 0.8598 |
| 5 | 39 / 0.8251 | 29 / 0.9247 | 250 / 0.8376 |
| 6 | 8 / 0.8659 | 16 / 0.8141 | 108 / 0.8603 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 101 / 0.3998 | 85 / 0.7079 | 416 / 0.8954 |
| 2 | 112 / 0.4771 | 164 / 0.7518 | 644 / 0.9231 |
| 3 | 91 / 0.5106 | 100 / 0.7230 | 474 / 0.9159 |
| 4 | 75 / 0.5926 | 59 / 0.7160 | 420 / 0.9431 |
| 5 | 39 / 0.5080 | 29 / 0.6713 | 250 / 0.9415 |
| 6 | 8 / 0.3770 | 16 / 0.6316 | 108 / 0.9503 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.7211 |
| Balanced accuracy | 0.6996 |
| Macro F1 | 0.6352 |
| Weighted F1 | 0.7394 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.3856 | 0.5892 | 0.4661 | 426 |
| malignant | 0.5278 | 0.7748 | 0.6279 | 453 |
| non-neoplastic | 0.9061 | 0.7349 | 0.8116 | 2312 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 251 | 61 | 114 |
| malignant | 40 | 351 | 62 |
| non-neoplastic | 360 | 253 | 1699 |


## 6. Segmentation experiments — train classifiers on cv2- and SAM2-segmented images

To isolate the effect of segmentation-aware preprocessing, run two more classifiers with the **same hyperparameters** as section 4 — only `--image_dir` changes:

- **cv2 variant** — ROI selection driven by LAB color variation, no SAM model. Fast.
- **SAM2 variant** — ROI selection constrained to SAM2's foreground mask (segmentation-aware crop).

Both are produced at **224×224** so the trainer's transform doesn't have to upscale at load time (which throws away mid-frequency texture detail).

Workflow:
1. Configure paths and segmentation knobs
2. Build the cv2-segmented dir
3. Install SAM2 + download a checkpoint
4. Build the SAM2-segmented dir
5. (Optional) Backfill SAM2 failures with cv2 so all three runs see the same md5s
6. Common training args
7. Train cv2 → train SAM2
8. Evaluate both

In [9]:
# Local-disk dirs for fast IO during training; mirrored back to Drive at the end.
LOCAL_CV2_DIR     = Path("/content/local_images_cv2_224")
LOCAL_SAM2_DIR    = Path("/content/local_images_sam2_224")
LOCAL_SAM3_DIR    = Path("/content/local_images_sam3_224")
LOCAL_MEDSAM3_DIR = Path("/content/local_images_medsam3_224")
DRIVE_CV2_DIR     = PROJECT_ROOT / "dataset/images_cv2_224"
DRIVE_SAM2_DIR    = PROJECT_ROOT / "dataset/images_sam2_224"
DRIVE_SAM3_DIR    = PROJECT_ROOT / "dataset/images_sam3_224"
DRIVE_MEDSAM3_DIR = PROJECT_ROOT / "dataset/images_medsam3_224"

# Segmentation knobs (apply to all backends)
SEG_SIZE  = 384      # SAM + ROI search resolution (downscaled to OUT_SIZE after)
OUT_SIZE  = 224      # final image side; matches the trainer's --image_size
CROP_FRAC = 0.6
MIN_SKIN  = 0.85

(PROJECT_ROOT / "logs").mkdir(parents=True, exist_ok=True)
for d in (LOCAL_CV2_DIR, LOCAL_SAM2_DIR, LOCAL_SAM3_DIR, LOCAL_MEDSAM3_DIR,
          DRIVE_CV2_DIR, DRIVE_SAM2_DIR, DRIVE_SAM3_DIR, DRIVE_MEDSAM3_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("CV2     →", DRIVE_CV2_DIR)
print("SAM2    →", DRIVE_SAM2_DIR)
print("SAM3    →", DRIVE_SAM3_DIR)
print("MedSAM3 →", DRIVE_MEDSAM3_DIR)
print("Source images at:", LOCAL_IMAGE_DIR, f"({_count_files(LOCAL_IMAGE_DIR)} files)")

CV2     → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_cv2_224
SAM2    → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam2_224
SAM3    → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam3_224
MedSAM3 → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_medsam3_224
Source images at: /content/local_images (16518 files)


### 6.1 cv2 segmentation (no SAM, fast)

Builds `images_cv2_224/` from the local cache. Runs in a few minutes on CPU. Resumable.

In [18]:
!python src/preprocess_segmentation.py \
    --backend cv2 \
    --strategy color \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_CV2_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_cv2_224.log'}"

# Mirror to Drive so it survives a Colab disconnect
!mkdir -p "{DRIVE_CV2_DIR}" && rsync -a "{LOCAL_CV2_DIR}/" "{DRIVE_CV2_DIR}/"
print("cv2 dir:",
      _count_files(LOCAL_CV2_DIR), "files local /",
      _count_files(DRIVE_CV2_DIR), "on Drive")

Backend:      cv2
Strategy:     color
seg_size:     384
out_size:     224
Input dir:    /content/local_images
Output dir:   /content/local_images_cv2_224
Total rows:   15956
Mask: cv2-only (all-ones fallback)
preprocess: 100%|██████████| 15956/15956 [05:24<00:00, 49.12it/s, fail=1, miss=0, ok=15955, skip=0]
[fail] a09323dec3002dbbb039e9cf06dcdb1f: image file is truncated (1 bytes not processed)

Done. processed=15955, skipped=0, missing_src=0, failed=1
Output: /content/local_images_cv2_224
Train against the new dir, e.g.:
  python src/train_baseline_efficientnet.py --image_dir /content/local_images_cv2_224 --image_size 224
cv2 dir: 15955 files local / 15955 on Drive


### 6.2 Install SAM2 and fetch the tiny checkpoint

`sam2` is not preinstalled on Colab. The checkpoint and config name must match — `sam2_hiera_tiny.pt` pairs with `sam2_hiera_t.yaml` (the config string is resolved by name inside the sam2 package).

In [19]:
SAM2_CKPT_DIR  = PROJECT_ROOT / "checkpoints"
SAM2_CKPT_PATH = SAM2_CKPT_DIR / "sam2_hiera_tiny.pt"
SAM2_CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not SAM2_CKPT_PATH.exists():
    !curl -L -o "{SAM2_CKPT_PATH}" \
        https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt
size_mb = SAM2_CKPT_PATH.stat().st_size / 1e6 if SAM2_CKPT_PATH.exists() else 0
print(f"SAM2 checkpoint: {SAM2_CKPT_PATH} (exists={SAM2_CKPT_PATH.exists()}, {size_mb:.1f} MB)")

# Install SAM2 itself (Meta's repo). Skip if already installed in this runtime.
try:
    import sam2  # noqa: F401
    print("sam2 already installed")
except ImportError:
    !pip install --quiet "git+https://github.com/facebookresearch/sam2.git"
    import sam2  # noqa: F401
    print("sam2 installed")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  148M  100  148M    0     0   104M      0  0:00:01  0:00:01 --:--:--  105M
SAM2 checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/checkpoints/sam2_hiera_tiny.pt (exists=True, 155.9 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.0 MB/s eta 0:00:00
sam2 installed


### 6.3 SAM2 segmentation (GPU)

Run the preprocessor with `--backend sam2`. ~30–45 min on a T4, ~10–15 min on an A100 for 16,519 images at `seg_size=384`. Resumable — files already in `--output_dir` are skipped.

The `tee` pipe captures the per-failure `[fail] <md5>: <exc>` lines, so you can audit which images SAM2 dropped this run.

In [20]:
!python src/preprocess_segmentation.py \
    --backend sam2 \
    --strategy color \
    --sam2_checkpoint "{SAM2_CKPT_PATH}" \
    --sam2_config sam2_hiera_t.yaml \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_SAM2_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} \
    --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

!mkdir -p "{DRIVE_SAM2_DIR}" && rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
print("SAM2 dir:",
      _count_files(LOCAL_SAM2_DIR), "files local /",
      _count_files(DRIVE_SAM2_DIR), "on Drive")

Backend:      sam2
Strategy:     color
seg_size:     384
out_size:     224
Input dir:    /content/local_images
Output dir:   /content/local_images_sam2_224
Total rows:   15956
Mask: sam2 loaded
preprocess:   0%|          | 0/15956 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sam2/sam2_image_predictor.py:431: UserWarning: cannot import name '_C' from 'sam2' (/usr/local/lib/python3.12/dist-packages/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  masks = self._transforms.postprocess_masks(
preprocess: 100%|██████████| 15956/15956 [35:27<00:00,  7.50it/s, fail=1, miss=0, ok=15955, skip=0]
[fail] a09323dec3002dbbb039e9cf06dcdb1f: image file is truncated (1 bytes not processed)

Done. processed=15955, skipped=0, mis

### 6.4 (Optional but recommended) Backfill SAM2 failures with cv2

`SkinLesionDataset` silently drops md5s whose `.jpg` is missing in `--image_dir`. If SAM2 fails on N images, the SAM2 trainer sees N fewer rows than the cv2 / raw trainers, and the seed-42 split produces a different cohort — breaking the comparison.

This cell re-runs the preprocessor with `--backend cv2` against the **same** `--output_dir`. Files already produced by SAM2 are skipped (resume-mode default), so this only fills in the gaps. After it finishes, `LOCAL_SAM2_DIR` should match `LOCAL_CV2_DIR`'s file count.

In [21]:
!python src/preprocess_segmentation.py \
    --backend cv2 \
    --strategy color \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_SAM2_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} 2>&1 | tee -a "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

!rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
print("SAM2 dir after backfill:",
      _count_files(LOCAL_SAM2_DIR), "files local /",
      _count_files(DRIVE_SAM2_DIR), "on Drive")

Backend:      cv2
Strategy:     color
seg_size:     384
out_size:     224
Input dir:    /content/local_images
Output dir:   /content/local_images_sam2_224
Total rows:   15956
Mask: cv2-only (all-ones fallback)
preprocess: 100%|██████████| 15956/15956 [00:00<00:00, 43427.40it/s, fail=1, miss=0, ok=0, skip=13747]
[fail] a09323dec3002dbbb039e9cf06dcdb1f: image file is truncated (1 bytes not processed)

Done. processed=0, skipped=15955, missing_src=0, failed=1
Output: /content/local_images_sam2_224
Train against the new dir, e.g.:
  python src/train_baseline_efficientnet.py --image_dir /content/local_images_sam2_224 --image_size 224
SAM2 dir after backfill: 15955 files local / 15955 on Drive


### 6.5 Install SAM3 and run text-prompted segmentation

Vanilla SAM3 with the text prompt `"skin lesion"`. The Python package `sam3` is from `facebookresearch/sam3`; weights are pulled by `build_sam3_image_model()` on first call. Output schema and `--seg_size` / `--out_size` are identical to the SAM2 run, so any downstream trainer code that worked on `images_sam2_224/` works on `images_sam3_224/`.

> If `pip install` from the SAM3 GitHub URL below fails, double-check the repo URL — the package is moving and the URL may have changed.

In [16]:
!pip uninstall numpy -y

Found existing installation: numpy 2.0.0
Uninstalling numpy-2.0.0:
  Successfully uninstalled numpy-2.0.0


In [10]:
!pip install "numpy<2,>=1.26"
try:
    import sam3  # noqa: F401
    print("sam3 already installed")
except ImportError:
    !pip install --quiet "git+https://github.com/facebookresearch/sam3.git"
    import sam3  # noqa: F401
    print("sam3 installed")


sam3 already installed


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [12]:
!python src/preprocess_segmentation.py \
    --backend sam3 \
    --strategy color \
    --prompt "skin lesion" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_SAM3_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} \
    --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam3_224.log'}"

!mkdir -p "{DRIVE_SAM3_DIR}" && rsync -a "{LOCAL_SAM3_DIR}/" "{DRIVE_SAM3_DIR}/"
print("SAM3 dir:",
      _count_files(LOCAL_SAM3_DIR), "files local /",
      _count_files(DRIVE_SAM3_DIR), "on Drive")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Backend:      sam3
Strategy:     color
seg_size:     384
out_size:     224
Input dir:    /content/local_images
Output dir:   /content/local_images_sam3_224
Total rows:   15956
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/facebook/sam3/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP

### 6.6 Install MedSAM3 and run LoRA-fine-tuned segmentation

`Joey-S-Liu/MedSAM3` = SAM3 + LoRA, fine-tuned on medical imagery. Two prerequisites the install cell sets up:

1. The repo cloned to `/content/MedSAM3` so `infer_sam.py` (which defines `SAM3LoRAInference`) is on `PYTHONPATH`.
2. The LoRA config (`configs/full_lora_config.yaml`) and trained weights (`outputs/sam3_lora_full/best_lora_weights.pt`) present inside the repo.

The trained weights are **not** committed to the MedSAM3 repo — follow that repo's README to download them, or copy them in from Drive.

In [23]:
MEDSAM3_DIR = Path("/content/MedSAM3")
if not MEDSAM3_DIR.exists():
    !git clone --depth 1 https://github.com/Joey-S-Liu/MedSAM3.git "{MEDSAM3_DIR}"
else:
    print(f"MedSAM3 already cloned at {MEDSAM3_DIR}")

# Optional: install MedSAM3's Python deps if its requirements file is present.
req = MEDSAM3_DIR / "requirements.txt"
if req.exists():
    !pip install --quiet -r "{req}"

MEDSAM3_CONFIG  = MEDSAM3_DIR / "configs/full_lora_config.yaml"
MEDSAM3_WEIGHTS = MEDSAM3_DIR / "outputs/sam3_lora_full/best_lora_weights.pt"

# Stash the LoRA weights on Drive so you don't re-download every session.
DRIVE_MEDSAM3_WEIGHTS = PROJECT_ROOT / "checkpoints/best_lora_weights.pt"
if not MEDSAM3_WEIGHTS.exists() and DRIVE_MEDSAM3_WEIGHTS.exists():
    MEDSAM3_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    !cp "{DRIVE_MEDSAM3_WEIGHTS}" "{MEDSAM3_WEIGHTS}"
    print(f"Restored LoRA weights from Drive → {MEDSAM3_WEIGHTS}")

print(f"Config:  {MEDSAM3_CONFIG} (exists={MEDSAM3_CONFIG.exists()})")
print(f"Weights: {MEDSAM3_WEIGHTS} (exists={MEDSAM3_WEIGHTS.exists()})")
assert MEDSAM3_CONFIG.exists(),  "Missing MedSAM3 LoRA config — see the MedSAM3 repo's README."
assert MEDSAM3_WEIGHTS.exists(), (
    "Missing MedSAM3 LoRA weights. Either download per the MedSAM3 README and place at "
    f"{MEDSAM3_WEIGHTS}, or upload to {DRIVE_MEDSAM3_WEIGHTS} on Drive and rerun this cell."
)

MedSAM3 already cloned at /content/MedSAM3
Config:  /content/MedSAM3/configs/full_lora_config.yaml (exists=True)
Weights: /content/MedSAM3/outputs/sam3_lora_full/best_lora_weights.pt (exists=True)


In [20]:
!ls /usr/local/lib/python3.12/dist-packages/sam3/assets/

bpe_simple_vocab_16e6.txt.gz


In [29]:
!cd /usr/local/lib/python3.12/dist-packages && \
  PYTHONPATH="{MEDSAM3_DIR}:{PROJECT_ROOT}" \
  python "{PROJECT_ROOT}/src/preprocess_segmentation.py" \
    --backend medsam3 \
    --strategy color \
    --prompt "skin lesion" \
    --medsam3_config "{MEDSAM3_CONFIG}" \
    --medsam3_weights "{MEDSAM3_WEIGHTS}" \
    --medsam3_resolution 1008 \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_MEDSAM3_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} \
    --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_medsam3_224.log'}"

!mkdir -p "{DRIVE_MEDSAM3_DIR}" && rsync -a "{LOCAL_MEDSAM3_DIR}/" "{DRIVE_MEDSAM3_DIR}/"
print("MedSAM3 dir:",
      _count_files(LOCAL_MEDSAM3_DIR), "files local /",
      _count_files(DRIVE_MEDSAM3_DIR), "on Drive")

Backend:      medsam3
Strategy:     color
seg_size:     384
out_size:     224
Input dir:    /content/local_images
Output dir:   /content/local_images_medsam3_224
Total rows:   15956
🔧 Initializing SAM3 + LoRA...
   Device: cuda
   Resolution: 1008x1008
   Confidence threshold: 0.3
   NMS IoU threshold: 0.5

📦 Building SAM3 model...
🔗 Applying LoRA configuration...
Replaced 61 nn.MultiheadAttention modules with MultiheadAttentionLoRA
Applied LoRA to 458 modules:
  - backbone.vision_backbone.trunk.blocks.0.attn.qkv
  - backbone.vision_backbone.trunk.blocks.0.attn.proj
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.1.attn.qkv
  - backbone.vision_backbone.trunk.blocks.1.attn.proj
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.2.attn.qkv
  - backbone.vision_backbone.trunk.blocks.2.attn.proj
  - b

### 6.7 Common training args

These match the section 4 baseline run (`20260427_055032`) **exactly**. The only thing that changes between the three runs is `--image_dir`.

In [30]:
# Mirror section 4 — keep these in sync if you change section 4.
SEG_IMAGE_SIZE   = 224
SEG_EPOCHS       = 40
SEG_BATCH_SIZE   = 32
SEG_LR           = 1e-4
SEG_WEIGHT_DECAY = 1e-4
SEG_NUM_WORKERS  = 4
SEG_OUTPUT_DIR   = OUTPUT_DIR     # all three runs share this parent; each gets its own timestamped subdir
print("Outputs land under:", SEG_OUTPUT_DIR)

Outputs land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


### 6.8 Train classifier on cv2-segmented images

Identical hyperparameters to the section 4 baseline (`20260427_055032`); only `--image_dir` changes.

In [23]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train batches: 399, Val batches: 100
Pretrained: True
Freeze backbone: True
model.safetensors: 100% 28.8M/28.8M [00:01<00:00, 25.5MB/s]
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.478, 2.4751, 0.4561]
Epoch [1/40] train_loss=1.0542, train_acc=0.4419, val_loss=1.0340, val_acc=0.4137  ← best
Epoch [2/40] train_loss=1.0034, train_acc=0.5162, val_loss=1.0081, val_acc=0.4510  ← best
Epoch [3/40] train_loss=0.9791, train_acc=0.5379, val_loss=0.9896, val_acc=0.4820  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9536, train_acc=0.5393, val_loss=0.9619, val_acc=0.5274  ← best
Epoch [5/40] train_loss=0.9215, train_acc=0.5714, val_loss=0.9363, val_acc=0.5719  ← best
Epoch [6/40] train_loss=0.8893, train_acc=0.5963, val_loss=0.9212, val_acc=0.5594  ← best
Epoch [7/40] train_loss=0.8641, train_acc=0.6141, val_loss=0.8992, val_acc=0.6073  ← best
Epoc

### 6.9 Train classifier on SAM2-segmented images

Same args as above, just `--image_dir` points at `LOCAL_SAM2_DIR`.

In [24]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train batches: 399, Val batches: 100
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.478, 2.4751, 0.4561]
Epoch [1/40] train_loss=1.0556, train_acc=0.4404, val_loss=1.0335, val_acc=0.4134  ← best
Epoch [2/40] train_loss=1.0045, train_acc=0.5210, val_loss=1.0078, val_acc=0.4494  ← best
Epoch [3/40] train_loss=0.9776, train_acc=0.5378, val_loss=0.9891, val_acc=0.4854  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9549, train_acc=0.5434, val_loss=0.9621, val_acc=0.5262  ← best
Epoch [5/40] train_loss=0.9239, train_acc=0.5718, val_loss=0.9385, val_acc=0.5575  ← best
Epoch [6/40] train_loss=0.8903, train_acc=0.5993, val_loss=0.9209, val_acc=0.5516  ← best
Epoch [7/40] train_loss=0.8633, train_acc=0.6163, val_loss=0.9006, val_acc=0.6086  ← best
Epoch [8/40] train_loss=0.8326, train_acc=0.6380, val_loss=0.883

### 6.10 Train classifier on SAM3-segmented images

Same args as the SAM2 run, just `--image_dir` points at `LOCAL_SAM3_DIR`.

In [ ]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM3_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

### 6.11 Train classifier on MedSAM3-segmented images

Same args as the SAM2 run, just `--image_dir` points at `LOCAL_MEDSAM3_DIR`.

In [31]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_MEDSAM3_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train batches: 399, Val batches: 100
Pretrained: True
Freeze backbone: True
model.safetensors: 100% 28.8M/28.8M [00:01<00:00, 20.3MB/s]
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.4567, 2.4956, 0.4562]
Epoch [1/40] train_loss=1.0569, train_acc=0.4192, val_loss=1.0236, val_acc=0.4256  ← best
Epoch [2/40] train_loss=1.0064, train_acc=0.5014, val_loss=0.9942, val_acc=0.4964  ← best
Epoch [3/40] train_loss=0.9812, train_acc=0.5203, val_loss=0.9773, val_acc=0.4983  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9566, train_acc=0.5480, val_loss=0.9511, val_acc=0.5218  ← best
Epoch [5/40] train_loss=0.9258, train_acc=0.5670, val_loss=0.9284, val_acc=0.5168  ← best
Epoch [6/40] train_loss=0.8982, train_acc=0.5949, val_loss=0.9070, val_acc=0.5572  ← best
Epoch [7/40] train_loss=0.8698, train_acc=0.6121, val_loss=0.8861, val_acc=0.5832  ← best
Epo

### 6.12 Evaluate all four segmented runs

Each cell below picks the most recent run whose `args.image_dir` matches the segmented dir, then writes `logs/<run_name>_report.md` and renders it inline. Run them in order — `evaluate.py` overwrites `logs/evaluation_metrics.json` each call, but the per-run markdown report is keyed by run name so all four are kept.

In [32]:
import torch as _torch_for_seg

def _seg_latest_under(image_dir: Path, label: str) -> Path:
    """Pick the most recent run under SEG_OUTPUT_DIR whose args.image_dir == image_dir."""
    runs = sorted(SEG_OUTPUT_DIR.glob("*/checkpoint.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    for ckpt in runs:
        try:
            args = _torch_for_seg.load(ckpt, map_location="cpu", weights_only=False).get("args", {})
        except Exception:
            continue
        if Path(args.get("image_dir", "")) == image_dir:
            print(f"[{label}] {ckpt}")
            return ckpt
    raise FileNotFoundError(f"No checkpoint found for image_dir={image_dir}")

CV2_CKPT = _seg_latest_under(LOCAL_CV2_DIR, "cv2")

!python src/evaluate.py \
    --checkpoint "{CV2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

cv2_report = PROJECT_ROOT / "logs" / f"{CV2_CKPT.parent.name}_report.md"
print("cv2 report:", cv2_report)
display(Markdown(cv2_report.read_text()))

[cv2] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260428_023527/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260428_023527/checkpoint.pt
Evaluating on 3191 samples (100 batches)
Traceback (most recent call last):
  File "/content/drive/MyDrive/SkinLesionBiasReduction/src/evaluate.py", line 184, in <module>
    main()
  File "/content/drive/MyDrive/SkinLesionBiasReduction/src/evaluate.py", line 158, in main
    y_true, y_pred, y_prob, skin_tones = collect_predictions(model, loader, device)
                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/SkinLesionBiasReduction/src/baseline_metrics.py", line 37, in collect_predictions
    for st

NameError: name 'Markdown' is not defined

In [27]:
SAM2_CKPT = _seg_latest_under(LOCAL_SAM2_DIR, "sam2")

!python src/evaluate.py \
    --checkpoint "{SAM2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

sam2_report = PROJECT_ROOT / "logs" / f"{SAM2_CKPT.parent.name}_report.md"
print("SAM2 report:", sam2_report)
display(Markdown(sam2_report.read_text()))

[sam2] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260428_025319/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260428_025319/checkpoint.pt
Evaluating on 3191 samples (100 batches)
Evaluation Summary
Samples: 3191
Top-1 Accuracy: 0.6594
Macro AUROC:    0.8142
Macro AUPRC:    0.6559

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=548): acc=0.5967 macroAUROC=0.7685 macroAUPRC=0.6089
             benign: n=   77 AUROC=0.7206 AUPRC=0.3265
          malignant: n=   93 AUROC=0.8103 AUPRC=0.6510
     non-neoplastic: n=  378 AUROC=0.7746 AUPRC=0.8492
fitzpatrick_2 (n=970): acc=0.6536 macroAUROC=0.8109 macroAUPRC=0.6627
             benign: n=  134 AUROC=0.7413 AUPRC=0.4113
          malignant: n=  157 AUROC=0.8659 AUPRC=0.6753
     non-neoplastic: n=  679 AUROC=0.8255 AUPRC=0.9013
fitzpatrick_3 (n=666): acc=0.65

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260428_025319/checkpoint.pt` |
| Split | val |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_cleaned.csv` |
| Image dir | `/content/local_images_sam2_224` |
| Generated at | 2026-04-28T03:17:08.256397 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 3191 |
| Top-1 Accuracy | 0.6594 |
| Macro AUROC | 0.8142 |
| Macro AUPRC | 0.6559 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 548 | 0.5967 | 0.7685 | 0.6089 |
| 2 | 970 | 0.6536 | 0.8109 | 0.6627 |
| 3 | 666 | 0.6577 | 0.8294 | 0.6755 |
| 4 | 564 | 0.7092 | 0.8352 | 0.6790 |
| 5 | 316 | 0.7184 | 0.8503 | 0.6834 |
| 6 | 127 | 0.6142 | 0.6987 | 0.5418 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 77 / 0.7206 | 93 / 0.8103 | 378 / 0.7746 |
| 2 | 134 / 0.7413 | 157 / 0.8659 | 679 / 0.8255 |
| 3 | 101 / 0.8056 | 90 / 0.8699 | 475 / 0.8128 |
| 4 | 75 / 0.8106 | 63 / 0.8658 | 426 / 0.8292 |
| 5 | 48 / 0.7633 | 22 / 0.9671 | 246 / 0.8206 |
| 6 | 6 / 0.5647 | 14 / 0.8502 | 107 / 0.6813 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 77 / 0.3265 | 93 / 0.6510 | 378 / 0.8492 |
| 2 | 134 / 0.4113 | 157 / 0.6753 | 679 / 0.9013 |
| 3 | 101 / 0.4866 | 90 / 0.6337 | 475 / 0.9064 |
| 4 | 75 / 0.5024 | 63 / 0.6051 | 426 / 0.9295 |
| 5 | 48 / 0.4280 | 22 / 0.7081 | 246 / 0.9140 |
| 6 | 6 / 0.0761 | 14 / 0.6554 | 107 / 0.8940 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6594 |
| Balanced accuracy | 0.6434 |
| Macro F1 | 0.5741 |
| Weighted F1 | 0.6855 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.3206 | 0.5533 | 0.4060 | 441 |
| malignant | 0.4532 | 0.7062 | 0.5521 | 439 |
| non-neoplastic | 0.8877 | 0.6707 | 0.7641 | 2311 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 244 | 66 | 131 |
| malignant | 64 | 310 | 65 |
| non-neoplastic | 453 | 308 | 1550 |


In [ ]:
SAM3_CKPT = _seg_latest_under(LOCAL_SAM3_DIR, "sam3")

!python src/evaluate.py \
    --checkpoint "{SAM3_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM3_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

sam3_report = PROJECT_ROOT / "logs" / f"{SAM3_CKPT.parent.name}_report.md"
print("sam3 report:", sam3_report)
display(Markdown(sam3_report.read_text()))

In [ ]:
MEDSAM3_CKPT = _seg_latest_under(LOCAL_MEDSAM3_DIR, "medsam3")

!python src/evaluate.py \
    --checkpoint "{MEDSAM3_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_MEDSAM3_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

medsam3_report = PROJECT_ROOT / "logs" / f"{MEDSAM3_CKPT.parent.name}_report.md"
print("medsam3 report:", medsam3_report)
display(Markdown(medsam3_report.read_text()))

## 7. Quick smoke test (optional)

If you want to verify the pipeline before committing to a full 40-epoch run, temporarily override section **4** with:

```python
IMAGE_SIZE = 64
EPOCHS     = 1
BATCH_SIZE = 128
```

Then re-run sections **4** and **5**. Once it completes without errors, restore the defaults (224 / 40 / 32) and launch the real training.

## 8. Optional — train the cGAN later

The generative side lives in `src/train.py` (vanilla cGAN; the WGAN-GP critic exists in `src/cgan.py` but the WGAN trainer hasn't been committed yet). To run on Colab once you're ready:

In [ ]:
# Uncomment when you want to start cGAN training. Outputs land under outputs/<timestamp>/.
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{IMAGE_DIR}" \
#     --epochs 200 \
#     --batch_size 64 \
#     --lr 0.0002 \
#     --output_dir "{PROJECT_ROOT / 'outputs'}" \
#     --device cuda

In [ ]:
# After (or during) cGAN training, watch losses + sample grids in TensorBoard:
# %load_ext tensorboard
# %tensorboard --logdir $PROJECT_ROOT/outputs